In [5]:
import pandas as pd
import numpy as np
import zipfile
import xml.etree.ElementTree as ET
import openpyxl

# 파일 경로
INPUT_FILE  = r"C:\유비온프로젝트2\corporate-bankruptcy\데이터수집\산업평균(에코스)\자산_자본 회전율(제11차 한국표준산업분류, 2009~)_27190748.xlsx"
OUTPUT_FILE = "자산_자본_회전율_지표_처리결과.xlsx"

# ─────────────────────────────────────────────
# 1. 업종 그룹 정의 (코드 → (그룹코드, 산업명))
# ─────────────────────────────────────────────
INDUSTRY_MAP = {
    "C10": ("C10, C11",  "음식료품 제조업"),
    "C11": ("C10, C11",  "음식료품 제조업"),
    "C13": ("C13, C14, C15", "섬유, 가죽, 신발 제조업"),
    "C14": ("C13, C14, C15", "섬유, 가죽, 신발 제조업"),
    "C15": ("C13, C14, C15", "섬유, 가죽, 신발 제조업"),
    "C16": ("C16, C17", "목재, 펄프, 종이 제조업"),
    "C17": ("C16, C17", "목재, 펄프, 종이 제조업"),
    "C20": ("C20, C21, C22", "화학물질, 화학제품, 의약품, 고무, 플라스틱 제조업"),
    "C21": ("C20, C21, C22", "화학물질, 화학제품, 의약품, 고무, 플라스틱 제조업"),
    "C22": ("C20, C21, C22", "화학물질, 화학제품, 의약품, 고무, 플라스틱 제조업"),
    "C23": ("C23",       "비금속광물제품 제조업"),
    "C24": ("C24",       "제1차 금속산업"),
    "C25": ("C25",       "조립금속제품 제조업"),
    "C29": ("C29",       "기타 기계 및 장비 제조업"),
    "C26": ("C26, C28",  "전자부품, 컴퓨터, 전기장비 제조업"),
    "C28": ("C26, C28",  "전자부품, 컴퓨터, 전기장비 제조업"),
    "C27": ("C27",       "의료, 정밀, 광학기기 및 시계 제조업"),
    "C30": ("C30, C31",  "운송장비 제조업"),
    "C31": ("C30, C31",  "운송장비 제조업"),
    "C32": ("C32, C33",  "기타 제품 제조업"),
    "C33": ("C32, C33",  "기타 제품 제조업"),
    "D35": ("D35",       "전기, 가스 및 수도사업"),
    "F":   ("F",         "건설업"),
    "G":   ("G",         "도매 및 소매업"),
    "I":   ("I",         "숙박 및 음식점업"),
    "H":   ("H",         "운수 및 창고업"),
    "J61": ("J61, J62, J63", "정보통신업"),
    "J62": ("J61, J62, J63", "정보통신업"),
    "J63": ("J61, J62, J63", "정보통신업"),
    "L":   ("L, M, N",   "부동산, 임대업, 사업서비스업"),
    "M":   ("L, M, N",   "부동산, 임대업, 사업서비스업"),
    "N":   ("L, M, N",   "부동산, 임대업, 사업서비스업"),
    "R":   ("R, S95, S96", "오락, 문화 및 수리·개인서비스업"),
    "S95": ("R, S95, S96", "오락, 문화 및 수리·개인서비스업"),
    "S96": ("R, S95, S96", "오락, 문화 및 수리·개인서비스업"),
}

# ─────────────────────────────────────────────────────────────────────────
# 2. xlsx를 zip으로 직접 파싱 (openpyxl 스타일 버그 완전 우회)
# ─────────────────────────────────────────────────────────────────────────
def read_xlsx_raw(filepath, sheet_name="데이터"):
    ns = {"ss": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(filepath) as z:
        shared_strings = []
        if "xl/sharedStrings.xml" in z.namelist():
            tree = ET.parse(z.open("xl/sharedStrings.xml"))
            for si in tree.findall(".//ss:si", ns):
                texts = [t.text or "" for t in si.findall(".//ss:t", ns)]
                shared_strings.append("".join(texts))

        wb_tree  = ET.parse(z.open("xl/workbook.xml"))
        rel_tree = ET.parse(z.open("xl/_rels/workbook.xml.rels"))
        rid_to_path = {
            r.attrib["Id"]: "xl/" + r.attrib["Target"].lstrip("/")
            for r in rel_tree.findall("*")
        }
        ns_wb = "http://schemas.openxmlformats.org/spreadsheetml/2006/main"
        ns_r  = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
        sheet_path = None
        for sh in wb_tree.findall(f".//{{{ns_wb}}}sheet"):
            if sh.attrib.get("name") == sheet_name:
                rid = sh.attrib.get(f"{{{ns_r}}}id")
                sheet_path = rid_to_path.get(rid)
                break
        if sheet_path is None:
            raise ValueError(f"시트 '{sheet_name}'를 찾을 수 없습니다.")

        ws_tree = ET.parse(z.open(sheet_path))
        rows_data = []
        for row in ws_tree.findall(".//ss:row", ns):
            row_cells = {}
            for cell in row.findall("ss:c", ns):
                ref   = cell.attrib.get("r", "")
                ctype = cell.attrib.get("t", "")
                v_el  = cell.find("ss:v", ns)
                val   = None
                if v_el is not None and v_el.text is not None:
                    if ctype == "s":
                        val = shared_strings[int(v_el.text)]
                    elif ctype == "b":
                        val = bool(int(v_el.text))
                    else:
                        try:
                            fval = float(v_el.text)
                            val  = int(fval) if fval == int(fval) else fval
                        except ValueError:
                            val = v_el.text
                col_str = "".join(c for c in ref if c.isalpha())
                row_cells[col_str] = val
            rows_data.append(row_cells)

    def col_to_idx(col):
        idx = 0
        for ch in col:
            idx = idx * 26 + (ord(ch) - ord("A") + 1)
        return idx - 1

    all_cols = sorted({k for row in rows_data for k in row}, key=col_to_idx)
    matrix   = [[row.get(c) for c in all_cols] for row in rows_data]
    return pd.DataFrame(matrix[1:], columns=matrix[0])

# ─────────────────────────────────────────────
# 3. 데이터 읽기
# ─────────────────────────────────────────────
df = read_xlsx_raw(INPUT_FILE, sheet_name="데이터")
print(f"원본 행수: {len(df)}, 열수: {len(df.columns)}")
print("컬럼:", df.columns.tolist())

# ─────────────────────────────────────────────
# 4. 불필요한 컬럼 제거
# ─────────────────────────────────────────────
drop_cols = ["통계표", "코드(기업규모)", "기업규모", "코드(계정항목)"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# ─────────────────────────────────────────────
# 5. 업종코드 → 그룹코드 + 산업명 매핑
# ─────────────────────────────────────────────
df["_raw_code"] = df["코드(업종코드)"].apply(lambda v: str(v).strip() if pd.notna(v) else "")
df["산업명"]    = df["_raw_code"].map(lambda c: INDUSTRY_MAP.get(c, (c, c))[1])
df["그룹코드"]  = df["_raw_code"].map(lambda c: INDUSTRY_MAP.get(c, (c, c))[0])
df = df.drop(columns=["코드(업종코드)", "업종코드", "_raw_code"])

# ─────────────────────────────────────────────
# 6. 연도 컬럼 식별 및 숫자 변환
# ─────────────────────────────────────────────
year_cols     = [c for c in df.columns if str(c).isdigit()]
non_year_cols = [c for c in df.columns
                 if not str(c).isdigit() and c not in ("산업명", "그룹코드")]
print(f"\n연도 컬럼: {year_cols}")
print(f"기타 컬럼: {non_year_cols}")

for yr in year_cols:
    df[yr] = pd.to_numeric(df[yr], errors="coerce")

# ─────────────────────────────────────────────
# 7. 그룹코드 + 계정항목 기준 집계 (연도별 평균)
# ─────────────────────────────────────────────
group_keys = ["그룹코드", "산업명", "계정항목"]
meta_cols  = [c for c in non_year_cols if c != "계정항목"]
agg_dict   = {yr: "mean" for yr in year_cols}
agg_dict.update({mc: "first" for mc in meta_cols})

df_grouped = df.groupby(group_keys, as_index=False).agg(agg_dict)

# ─────────────────────────────────────────────
# 8. 컬럼 순서 정리 및 이름 변경
# ─────────────────────────────────────────────
ordered_cols = ["산업명", "그룹코드", "계정항목"] + meta_cols + year_cols
df_grouped   = df_grouped[[c for c in ordered_cols if c in df_grouped.columns]]
df_grouped   = df_grouped.rename(columns={"그룹코드": "코드(업종코드)"})

# ─────────────────────────────────────────────
# 9. 반올림
# ─────────────────────────────────────────────
df_grouped[year_cols] = df_grouped[year_cols].round(2)

# ─────────────────────────────────────────────
# 10. openpyxl로 직접 저장 (pandas 버전 무관)
# ─────────────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "결과"

# 헤더
ws.append(df_grouped.columns.tolist())

# 데이터 행
for row in df_grouped.itertuples(index=False):
    ws.append(list(row))

wb.save(OUTPUT_FILE)

print(f"\n✅ 완료! 결과 파일: {OUTPUT_FILE}")
print(f"최종 행수: {len(df_grouped)}, 열수: {len(df_grouped.columns)}")
print("\n최종 컬럼:", df_grouped.columns.tolist())
print("\n샘플 (첫 5행):")
print(df_grouped.head())

원본 행수: 350, 열수: 22
컬럼: ['통계표', '코드(업종코드)', '업종코드', '코드(기업규모)', '기업규모', '코드(계정항목)', '계정항목', '단위', '변환', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

연도 컬럼: ['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
기타 컬럼: ['계정항목', '단위', '변환']

✅ 완료! 결과 파일: 자산_자본_회전율_지표_처리결과.xlsx
최종 행수: 200, 열수: 18

최종 컬럼: ['산업명', '코드(업종코드)', '계정항목', '단위', '변환', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

샘플 (첫 5행):
        산업명  코드(업종코드)      계정항목  단위   변환   2012   2013   2014   2015   2016  \
0  음식료품 제조업  C10, C11   경영자산회전율  회   원자료   1.14   1.12   1.12   1.11   1.10   
1  음식료품 제조업  C10, C11   매입채무회전율  회   원자료  15.43  15.35  15.60  16.03  15.50   
2  음식료품 제조업  C10, C11   매출채권회전율  회   원자료   8.18   7.89   7.92   8.15   7.94   
3  음식료품 제조업  C10, C11  비유동자산회전율  회   원자료   1.50   1.47   1.46   1.44   1.42   
4  음식료품 제조업  C10, C11  상(제)품회전율  회   